# Figure 2 - GNN-Pearson Architecture: Forward Pass and Information Flow

A schematic of a left-to-right schematic of the model's forward pass:

**Input Graph** (40 nodes, $X(t)\in\mathbb{R}^{40\times24}$)  →  **3 stacked GCN layers**
(hidden 128, ReLU, DropEdge, residual skips)  →  **GDP-node readout** ($h_{GDP}^{(3)}\in\mathbb{R}^{128}$)
→  **MLP head** (FC 128→64 ReLU, FC 64→1 linear)  →  **output** $\hat{g}_{t+1}$ (SAAR %).

Every label is drawn from the trained model in `V72_Final.ipynb` and is verified consistent with it:

| Element | Value | Source in `V72_Final.ipynb` |
|---|---|---|
| Hidden dim | 128 | `PARAMS["hidden_channels"] = 128` |
| GCN layers | 3 (`conv1/2/3`) | `GCNConv` x3 + `input_proj` residual |
| DropEdge | 10% (layers 1-2) | `PARAMS["drop_edge_rate"] = 0.1` |
| Feature channels | 4 x 6-month = 24 dims/node | levels, velocity, volatility, accel. |
| Ensembling | 5 seeds x 3 checkpoints = 15 preds, trimmed mean | `n_seeds=5`, `k_snapshots=3` |
| Loss | RegimeAwareSAARLoss, $\lambda=1.0$, scale $=5.0$, 3x sign-change weight | `direction_lambda=1.0`, `sign_scale=5.0` |

The figure is drawn purely with matplotlib primitives (rounded boxes, arrows, a mini ring graph)
and saved as `fig_2_architecture.png` in this same folder.

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle, Ellipse
from matplotlib.lines import Line2D
import numpy as np, pathlib

# ---------- palette ----------
C_FRED   = "#2c6fbb"   # blue  (macro / inner ring)
C_MRTS   = "#2ca02c"   # green (retail / outer ring)
C_HUB    = "#A8780A"   # gold-brown GDP hub
C_GCNbox = "#eef0f8"; C_GCNedge = "#8a93c8"; C_GCNtxt = "#3b4aa0"
C_RDfill = "#fdf3d0"; C_RDedge  = "#C9A227"; C_RDtxt  = "#8a6d0b"
C_MLPfill= "#ece6f5"; C_MLPedge = "#6a3d9a"; C_MLPtxt = "#5b2d8a"
C_OUTfill= "#d7ecd9"; C_OUTedge = "#2a7a35"; C_OUTtxt = "#1f6b2c"
C_LGfill = "#eef0fb"; C_LGedge  = "#b9c2e8"
C_LOSSf  = "#fdf6d8"; C_LOSSe   = "#f0a020"; C_LOSStitle = "#6b4a2b"; C_LOSStxt="#5a4632"
C_RES    = "#e8821e"   # orange residual
C_FLOW   = "#5a7a8a"   # flow arrows
C_HEAD   = "#444444"   # column header gray
C_NAVY   = "#1a2a8a"

FIGW, FIGH = 20, 9.3
ADJ = dict(left=0.01, right=0.99, bottom=0.02, top=0.90)
# x-radius shrink factor so equal-radius shapes render as TRUE circles (axes is wide, ranges equal)
ASP = ((ADJ["top"]-ADJ["bottom"])*FIGH) / ((ADJ["right"]-ADJ["left"])*FIGW)
fig, ax = plt.subplots(figsize=(FIGW, FIGH))
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off"); ax.set_aspect("auto")

def vcircle(x, y, r, fc, ec="white", lw=1.2, z=4):
    ax.add_patch(Ellipse((x, y), width=2*r*ASP, height=2*r,
                 facecolor=fc, edgecolor=ec, linewidth=lw, zorder=z))

def rbox(x, y, w, h, fc, ec, lw=2.0, rad=0.025, z=2, square=False):
    style = "square,pad=0" if square else f"round,pad=0,rounding_size={rad*100}"
    p = FancyBboxPatch((x, y), w, h, boxstyle=style,
                       facecolor=fc, edgecolor=ec, linewidth=lw, zorder=z,
                       mutation_aspect=h/w if w else 1)
    ax.add_patch(p); return p

def arrow(x1, y1, x2, y2, color=C_FLOW, lw=2.2, ls="-", rad=0.0, z=5, mut=18):
    a = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=mut,
                        color=color, lw=lw, linestyle=ls, zorder=z,
                        connectionstyle=f"arc3,rad={rad}", shrinkA=0, shrinkB=0)
    ax.add_patch(a)

def T(x, y, s, size=11, color="black", weight="normal", style="normal", ha="center", va="center", z=6, fam=None):
    ax.text(x, y, s, fontsize=size, color=color, fontweight=weight, fontstyle=style,
            ha=ha, va=va, zorder=z, family=fam)

# ---------- column separators ----------
for sx in [17, 59.5, 70, 85.5]:
    ax.plot([sx, sx], [4, 90], ls=(0,(4,4)), color="#cccccc", lw=1.1, zorder=1)

# ---------- column headers ----------
T(8.5, 91, "Input Graph", 13, C_HEAD, "bold")
T(38, 91, "Graph Convolutional Network   —   3 Stacked Layers", 13, C_HEAD, "bold")
T(64.5, 91, "Readout", 13, C_HEAD, "bold")
T(77.5, 91, "MLP Head", 13, C_HEAD, "bold")
T(92.5, 91, "Output", 13, C_HEAD, "bold")
T(8.5, 87.3, "$G=(V,E)$   |   40 nodes", 10.5, "#555555", "normal", "italic")

# ================= INPUT GRAPH: mini ring =================
gx, gy, RIN, ROUT = 8.5, 72, 5.0, 8.8
vcircle(gx, gy, 2.3, C_HUB, "white", 1.5, z=4)
T(gx, gy, "GDP\nHub", 8.5, "white", "bold", z=5)
rng = np.linspace(0, 2*np.pi, 13)[:-1]
inner_lbl = {0:"PCE", 3:"IP", 6:"M2", 9:"PPI"}
for i, a in enumerate(rng):
    px, py = gx + RIN*np.cos(a)*ASP, gy + RIN*np.sin(a)
    ax.plot([gx, px],[gy, py], color="#dddddd", lw=0.7, zorder=1)
    vcircle(px, py, 0.80, C_FRED, "white", 0.6, z=3)
    if i in inner_lbl: T(px+(1.5 if np.cos(a)>=0 else -1.5), py, inner_lbl[i], 7, C_FRED, "bold",
                          ha="left" if np.cos(a)>=0 else "right")
for i, a in enumerate(rng + 0.26):
    px, py = gx + ROUT*np.cos(a)*ASP, gy + ROUT*np.sin(a)
    vcircle(px, py, 0.80, C_MRTS, "white", 0.6, z=3)

# ----- legend box: node feature vector -----
lx, ly, lw_, lh = 1.0, 30, 15.2, 30
rbox(lx, ly, lw_, lh, C_LGfill, C_LGedge, lw=1.6, rad=0.02)
T(lx+lw_/2, ly+lh-3.2, "Node feature vector  $x_i(t)$", 11, C_NAVY, "bold")
chans = [("Levels","(z-scored)"),("Velocity","(1st diff)"),("Volatility","(3-mo std)"),("Accel.","(2nd diff)")]
cy0 = ly+lh-7.5
for k,(nm,desc) in enumerate(chans):
    yy = cy0 - k*3.7
    ax.add_patch(plt.Rectangle((lx+1.6, yy-0.55), 1.1, 1.1, facecolor=C_NAVY, edgecolor="none", zorder=5))
    T(lx+3.3, yy, nm, 10, C_NAVY, "bold", ha="left")
    T(lx+8.4, yy, desc, 9, "#666666", "normal", "italic", ha="left")
T(lx+lw_/2, ly+5.6, "6-month window × 4 channels = 24 dims / node", 8.7, "#555555","normal","italic")
T(lx+lw_/2, ly+2.4, "Input tensor  $X(t)\\in\\mathbb{R}^{40\\times24}$", 10.5, C_NAVY, "bold")

# ----- seeds box -----
rbox(lx, 22.5, lw_, 5.2, C_LGfill, C_LGedge, lw=1.4, rad=0.03)
T(lx+lw_/2, 25.1, "5 seeds × 3 checkpoints = 15 preds → trimmed mean", 8.4, "#444444","normal","italic")

# ================= GCN LAYERS =================
layer_cx = [27.5, 39.5, 51.5]
sub = ["1-hop aggregation","2-hop aggregation","3-hop aggregation"]
LW, LH, LY = 7.4, 34, 47          # layer container box (narrow, square corners)
node_cols = [C_MRTS, C_FRED, C_HUB, C_FRED, C_MRTS]
for li, cx in enumerate(layer_cx):
    x0 = cx - LW/2
    rbox(x0, LY, LW, LH, C_GCNbox, C_GCNedge, lw=1.8, square=True)
    T(cx, LY+LH+4.2, f"GCN Layer {li+1}", 11.5, C_GCNtxt, "bold")
    T(cx, LY+LH+1.4, sub[li], 9.3, C_GCNtxt, "normal", "italic")
    # nodes column (true circles + connecting arrows between them)
    nys = np.linspace(LY+LH-5, LY+5, 5)
    rad = lambda col: 2.15 if col==C_HUB else 1.8
    for j in range(4):
        r0, r1 = rad(node_cols[j]), rad(node_cols[j+1])
        cc = 0.30 if j % 2 == 0 else -0.30
        arrow(cx, nys[j]-r0-0.1, cx, nys[j+1]+r1+0.1, color="#7f88c2", lw=1.4, rad=cc, z=3, mut=9)
    for ny, col in zip(nys, node_cols):
        vcircle(cx, ny, rad(col), col, "white", 1.3, z=4)
    # bottom note box
    note = "Hidden dim: 128\nReLU activation" + ("\nDropEdge 10%" if li<2 else "")
    rbox(x0+0.4, 31.5, LW-0.8, 12.5, "#f4f5fb", C_GCNedge, lw=1.4, square=True)
    T(cx, 37.7, note, 9.3, C_GCNtxt, "normal")

# inter-layer message passing: faint same-row arrows + brown curves fanning into the next hub
_nys = np.linspace(LY+LH-5, LY+5, 5)
_rad = lambda col: 2.15 if col==C_HUB else 1.8
_huby = _nys[2]; _rH = _rad(C_HUB)
C_AGG = "#8a5a08"
for li in range(2):
    cxL, cxR = layer_cx[li], layer_cx[li+1]
    for j, ny in enumerate(_nys):
        r0 = _rad(node_cols[j])
        xL, xR = cxL + r0*ASP + 0.3, cxR - _rH*ASP - 0.5
        if j == 2:                                   # hub -> hub (straight, bold brown)
            arrow(cxL + _rH*ASP + 0.3, ny, xR, _huby, color=C_AGG, lw=1.6, mut=11, z=3)
        else:                                        # same-row faint gray + brown curve into hub
            arrow(xL, ny, cxR - r0*ASP - 0.5, ny, color="#cfd4e6", lw=0.9, mut=7, z=2)
            arrow(xL, ny, xR, _huby, color=C_AGG, lw=1.2, rad=(0.25 if ny > _huby else -0.25), mut=9, z=3)
# input arrows into layer 1 (from graph)
for yy in np.linspace(LY+LH-6, LY+7, 5):
    arrow(17.6, yy, layer_cx[0]-LW/2-0.2, yy, "#b9c0d8", 1.4, mut=11)
# residual skip arcs (orange, over the top)
arrow(layer_cx[0], LY+LH-1, layer_cx[1], LY+LH-1, C_RES, 2.0, rad=-0.45, z=5, mut=16)
arrow(layer_cx[1], LY+LH-1, layer_cx[2], LY+LH-1, C_RES, 2.0, rad=-0.45, z=5, mut=16)
T((layer_cx[0]+layer_cx[1])/2, LY+LH+0.2, "residual", 8, C_RES, "bold", "italic")
# DropEdge × marks at right edge of layers 1,2
for cx in layer_cx[:2]:
    for yy in np.linspace(LY+LH-8, LY+8, 3):
        T(cx+LW/2-0.4, yy, "×", 11, "#9aa0b0", "bold")

# ================= READOUT =================
arrow(layer_cx[2]+LW/2+0.2, LY+LH/2, 60.5, LY+LH/2, C_FLOW, 2.4, mut=18)
rbox(60.8, 55, 7.4, 12, C_RDfill, C_RDedge, lw=2.0, rad=0.05)
T(64.5, 62.4, "GDP Node\nReadout", 11, C_RDtxt, "bold")
T(64.5, 57.4, "$h_{GDP}^{(3)}\\in\\mathbb{R}^{128}$", 10, C_RDtxt)

# ================= MLP HEAD =================
arrow(68.3, 61, 70.8, 61, C_FLOW, 2.2, mut=15)
rbox(71.0, 55.5, 6.0, 11, C_MLPfill, C_MLPedge, lw=2.0, rad=0.05)
T(74.0, 62.0, "FC (128→64)", 10, C_MLPtxt, "bold"); T(74.0, 58.6, "ReLU", 9, C_MLPtxt,"normal","italic")
arrow(77.1, 61, 79.0, 61, C_FLOW, 2.2, mut=15)
rbox(79.2, 55.5, 6.0, 11, C_MLPfill, C_MLPedge, lw=2.0, rad=0.05)
T(82.2, 62.0, "FC (64→1)", 10, C_MLPtxt, "bold"); T(82.2, 58.6, "linear", 9, C_MLPtxt,"normal","italic")

# ================= OUTPUT =================
arrow(85.3, 61, 87.4, 61, C_OUTedge, 3.0, mut=20)
rbox(87.6, 54, 10.2, 14, C_OUTfill, C_OUTedge, lw=2.6, rad=0.05)
T(92.7, 63.0, "$\\hat{g}_{t+1}$", 17, C_OUTtxt, "bold")
T(92.7, 57.0, "GDP Growth\n(SAAR, %)", 9.5, C_OUTtxt, "bold")

# ================= LOSS BOX (green, matching the output box) =================
rbox(71.5, 8, 26, 24, C_OUTfill, C_OUTedge, lw=2.2, rad=0.04)
T(84.5, 27.5, "RegimeAwareSAARLoss", 12, C_OUTtxt, "bold")
T(84.5, 22.0, "$\\mathcal{L}=$ MAE $+\\ \\lambda\\cdot$ DirPenalty", 11, C_OUTtxt)
T(84.5, 17.0, "Sign-change quarters: 3×  weight", 10.5, C_OUTtxt)
T(84.5, 12.0, "$\\lambda=1.0,\\ \\ $ scale $=5.0$", 10.5, C_OUTtxt)
# dashed feedback arrow from output down to loss
arrow(92.7, 53.8, 90, 32.2, C_OUTedge, 1.6, ls=(0,(4,3)), rad=0.15, z=4, mut=13)

# ================= TITLE + SUBTITLE =================
ax.set_title("Figure 2.  GNN-Pearson Architecture: Forward Pass and Information Flow",
             fontsize=15.5, fontweight="bold", pad=26)
ax.text(0.5, 1.022,
        "Input graph $G$ with 40 nodes and feature tensor $X(t)\\in\\mathbb{R}^{40\\times24}$   "
        "→  3-layer GCN with residual skip connections and DropEdge  →  GDP node readout  "
        "→  MLP head  →  one-quarter-ahead SAAR forecast.",
        transform=ax.transAxes, ha="center", va="center", fontsize=10.3, style="italic", color="#555555")

fig.subplots_adjust(**ADJ)
FIG_DIR = pathlib.Path("./figures")
fig.savefig(FIG_DIR/"fig_2_architecture.png", dpi=150, bbox_inches="tight")
print("Saved:", FIG_DIR/"fig_2_architecture.png")


## Notes
- Layer containers are narrow with square corners (`square=True` in `rbox`); the loss box is colored green to match the output box (`C_OUT*`).
- Between layers: faint gray **same-row** arrows plus brown **aggregation curves** fanning from every node into the next layer's GDP hub (bold brown hub->hub), mirroring the doc's message-passing detail.
- Node markers are drawn as ellipses scaled by `ASP` so they render as **true circles** on the wide canvas; the small curved arrows inside each layer denote node-to-node message passing.
- Pure-schematic figure: no data load required - all dimensions/hyperparameters are literal labels
  matching `V72_Final.ipynb` (see the table above).
- Layout uses a 0-100 coordinate canvas; `layer_cx`, `LY/LH/LW`, and the per-section blocks
  control placement. Colors are grouped at the top (`C_*`).
- The bottom-left **node feature vector** legend and bottom-right **RegimeAwareSAARLoss** box
  mirror the doc; the orange arcs over the GCN stack denote the residual skip connections and
  the gray "x" marks denote DropEdge.
- Saved as `fig_2_architecture.png` (dpi 150) alongside this notebook.